In [1]:
# hide-output 

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

import pandas as pd
import numpy as np
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import plotly.graph_objects as go

# Stephen Few dataviz grounding (perceptualedge.com, "Practical Rules for Using Color in
# Charts") - light background, muted/desaturated colors for ordinary data, reserving anything
# warmer/more saturated for whatever should actually draw the eye (e.g. the overflow/farming
# zone). Same constants/spirit as abtestmetrics' common_lib.metrics
# (_PLOTLY_TEMPLATE/_CHART_BG_COLOR/_VARIANT_COLORS) - not reused directly since none of these
# charts are a Test-vs-control split, but grounded in the same source.
_FEW_TEMPLATE = 'plotly_white'
_FEW_BG = '#f7f7f5'
_FEW_QUALITATIVE = ['#2E5E67', '#6FA98C', '#B08968', '#8A7CA8', '#6E8894', '#A85751']
_LADDER_COLORS = {
    '1-16 (designed ladder)': '#5B7C99', '17+ (overflow)': '#C87941',
    'Ladder (1-16)': '#5B7C99', 'Overflow (17+)': '#C87941',
}


def few_style(fig):
    fig.update_layout(template=_FEW_TEMPLATE, plot_bgcolor=_FEW_BG, paper_bgcolor=_FEW_BG)
    return fig


def sequential_colors(n, scale='Blues', lo=0.35, hi=0.9):
    """n muted, graduated colors for an ORDERED categorical (e.g. Level 1..N, Stage 0..15) -
    Few's guidance to use a single-hue gradient for ordered data rather than a qualitative
    rainbow. lo/hi keep both ends off the colorscale's near-white/near-black extremes."""
    if n <= 1:
        return px.colors.sample_colorscale(scale, [hi])
    return px.colors.sample_colorscale(scale, [lo + (hi - lo) * i / (n - 1) for i in range(n)])


In [2]:
# hide-output 
refresh_data = False
bqc = BigQueryConnector()

In [3]:
# hide-output 
data = bqc.load_or_query('tempgen', refresh_data)

This query will process 2.03 GB when run.
Estimated query cost: $0.01


In [4]:
# hide-output 
#data

## Engagement

A way to determine who actually engaged with the feature we can us `rp_unnested_generator_activated_array`'s `generator_activated_object_path = 'liveops-GeneratorEvent-GardenGlory-GenPath'` (see [sql/generator_activated.sql](./sql/generator_activated.sql)) - a row means the generator existed on the player's board **and** was collected from at least once. There is no separate "enabled but never used" signal anywhere in the warehouse, so this is the closest available proxy for "had it on," not a literal on/off flag. This gives a fixed population of **26,363** players (after the standard `dim_users_to_exclude` filter) who we know for certain had the generator active, independent of nominal treatment-group size.

The three charts below track this **same 26,363-player cohort** across all three paths (generator/primary/secondary) - each shows, day by day, the cumulative count (and % of the fixed 26,363) who had reached that path at all (level >= 1) by that date.

In [5]:
# hide-output 
generator_activated = bqc.load_or_query('generator_activated', refresh_data)
generator_activated['first_activation_dt'] = pd.to_datetime(generator_activated['first_activation_dt'])

This query will process 2.05 GB when run.
Estimated query cost: $0.01


In [6]:
# hide-output 
#generator_activated

In [7]:
# The fixed denominator for all three charts below - every progression curve is measured against
# this same cohort, not each path's own (larger, for primary/secondary overlapping-but-not-
# identical) population, so the charts show how far THIS specific group progressed rather than
# three differently-sized populations.
generator_population = generator_activated['user_id'].unique()
n_population = len(generator_population)

data['dt'] = pd.to_datetime(data['dt'])

first_reached = {
    'generatorPath': generator_activated.set_index('user_id')['first_activation_dt'],
    'primaryPath': data[data['path_type'] == 'primaryPath'].groupby('user_id')['dt'].min(),
    'secondaryPath': data[data['path_type'] == 'secondaryPath'].groupby('user_id')['dt'].min(),
}

# Restrict to the fixed 26,363 cohort - a handful of primary/secondary discoveries belong to
# users with no captured generator-activation event (activation-event capture gaps, not a real
# progression path into primary/secondary without ever having the generator on), so they're
# excluded here rather than silently inflating counts against a denominator they're not part of.
first_reached = {
    path: series[series.index.isin(generator_population)]
    for path, series in first_reached.items()
}

for path in ['primaryPath', 'secondaryPath']:
    all_reachers = data[data['path_type'] == path]['user_id'].nunique()
    in_cohort = first_reached[path].shape[0]
    print(f'{path}: {all_reachers} total reachers, {in_cohort} within the 26,363 generator cohort ({all_reachers - in_cohort} excluded, no matching generator-activation event)')

day_range = pd.date_range('2026-08-17', '2026-08-30', freq='D')

def cumulative_reach_by_day(first_reached_dates: pd.Series, day_range: pd.DatetimeIndex, population_size: int) -> pd.DataFrame:
    """For each day, how many of population_size had already reached this path (first_reached_dates
    <= that day) - a monotonically non-decreasing adoption curve, not a snapshot of that day's
    activity."""
    counts = [(first_reached_dates <= day).sum() for day in day_range]
    df = pd.DataFrame({'dt': day_range, 'cumulative_users': counts})
    df['pct_of_population'] = (df['cumulative_users'] / population_size * 100).round(1)
    df['label'] = df['cumulative_users'].map('{:,}'.format) + ' (' + df['pct_of_population'].astype(str) + '%)'
    return df

reach_by_path = {
    path: cumulative_reach_by_day(dates, day_range, n_population)
    for path, dates in first_reached.items()
}

primaryPath: 26297 total reachers, 26291 within the 26,363 generator cohort (6 excluded, no matching generator-activation event)
secondaryPath: 13341 total reachers, 13341 within the 26,363 generator cohort (0 excluded, no matching generator-activation event)


In [8]:
path_labels = {'generatorPath': 'Generator', 'primaryPath': 'Primary', 'secondaryPath': 'Secondary'}

for path, label in path_labels.items():
    fig = px.bar(
        reach_by_path[path],
        x='dt',
        y='cumulative_users',
        text='label',
        title=f'{label} Path Adoption Over Time (cumulative, % of {n_population:,}-player generator cohort)',
        width=1200,
        height=400,
        labels={'cumulative_users': 'Cumulative Users', 'dt': 'Date'},
        hover_data={'pct_of_population': ':.1f', 'label': False},
        color_discrete_sequence=[_FEW_QUALITATIVE[0]],
    )
    fig.update_traces(textposition='outside')
    few_style(fig)
    fig.show()

### Engagement intensity

Adoption (above) is binary - did a player ever reach the path at all. This looks at *daily intensity* instead: what share of that day's Treatment-group DAU used the generator at least X times that specific day. Denominator is Treatment DAU specifically, not total game DAU - control never sees the generator at all, so the whole game's DAU would badly understate the real engagement rate among the population that could actually use it.

Reuses `abtestmetrics`' `abtest_assign_data.pkl`/`abtest_metrics.pkl` (same merge/exclusion pattern as elsewhere - pre-assignment rows routed to `NotAssigned`) purely to get a Treatment DAU count per day; no farmer-exclusion or metric-comparison machinery this time, just a denominator.

**`HEAVY_USAGE_THRESHOLDS` is a list** - every threshold in it gets computed and drawn as its own line on the same chart, so different intensity cuts (e.g. "used it at least once" vs "used it 10+ times") can be compared directly rather than one at a time.

In [9]:
# hide-output 
generator_activated_daily = bqc.load_or_query('generator_activated_daily', refresh_data)
generator_activated_daily['dt'] = pd.to_datetime(generator_activated_daily['dt'])

This query will process 2.05 GB when run.
Estimated query cost: $0.01


In [10]:
# Treatment DAU per day, reused from abtestmetrics - a row in abtest_metrics.pkl already means
# "active that day" (there's no explicit 0 row for inactive days), so counting distinct users
# per dt within the Test variant is the DAU count directly, no extra active-flag filter needed.
ab_assign = pd.read_pickle('../abtestmetrics/data/abtest_assign_data.pkl')
ab_data_metrics = pd.read_pickle('../abtestmetrics/data/abtest_metrics.pkl')
ab_data_metrics = ab_data_metrics.merge(ab_assign[['user_id', 'variant', 'assigned_dt']], on='user_id', how='left')
ab_data_metrics['variant'] = ab_data_metrics['variant'].fillna('NotAssigned')
ab_data_metrics.loc[ab_data_metrics['dt'] < ab_data_metrics['assigned_dt'], 'variant'] = 'NotAssigned'
ab_data_metrics['dt'] = pd.to_datetime(ab_data_metrics['dt'])

treatment_dau = ab_data_metrics[ab_data_metrics['variant'] == 'Test'].groupby('dt')['user_id'].nunique().rename('treatment_dau')

HEAVY_USAGE_THRESHOLDS = [1, 3, 10]  # any list of X-activations/day thresholds - each gets its own line on the chart below

heavy_usage_rows = []
for threshold in HEAVY_USAGE_THRESHOLDS:
    n_heavy_users = (
        generator_activated_daily[generator_activated_daily['n_activations'] >= threshold]
        .groupby('dt')['user_id'].nunique()
    )
    df = pd.DataFrame(index=day_range).join(n_heavy_users.rename('n_heavy_users')).join(treatment_dau)
    df['n_heavy_users'] = df['n_heavy_users'].fillna(0)
    df['pct_of_treatment_dau'] = (df['n_heavy_users'] / df['treatment_dau'] * 100).round(2)
    df = df.reset_index().rename(columns={'index': 'dt'})
    df['threshold_label'] = f'>= {threshold}x/day'
    heavy_usage_rows.append(df)

heavy_usage_pct = pd.concat(heavy_usage_rows, ignore_index=True)
threshold_order = [f'>= {t}x/day' for t in HEAVY_USAGE_THRESHOLDS]

In [11]:
fig = px.line(
    heavy_usage_pct, x='dt', y='pct_of_treatment_dau', color='threshold_label',
    category_orders={'threshold_label': threshold_order},
    color_discrete_sequence=sequential_colors(len(HEAVY_USAGE_THRESHOLDS)),
    markers=True,
    hover_data={'n_heavy_users': ':,.0f', 'treatment_dau': ':,'},
    title='% of Treatment DAU Using Generator >= X Times Daily',
    labels={'dt': 'Date', 'pct_of_treatment_dau': '% of Treatment DAU', 'threshold_label': 'Threshold'},
    width=1200,
    height=500,
)
few_style(fig)
fig.show()

### Current level distribution

The adoption charts above show *whether* a player reached a path, not *how far* they got. The two charts below (per path) add that: a **funnel view** - one line per level, each showing the cumulative count (running-max, forward-filled) of players who reached *at least* that level by each day, so a player at level 5 counts toward every line from level 1 through 5, not just their own level (unlike a stacked distribution, where each player would only count toward one exclusive segment) - and a median + P10-P90 band trend line, computed over the whole cohort (players who haven't reached the path yet count as level 0, which is why the median stays at 0 until a majority have started).

In [12]:
# hide-output 
def build_current_level_grid(path_type: str, population, day_range: pd.DatetimeIndex) -> pd.DataFrame:
    """Per-user, per-day current (running-max) level for path_type, forward-filled across days
    with no new discovery. cummax() alone does NOT forward-fill - it only skips NaN cells when
    computing later maxima but leaves the NaN itself in place - so ffill() has to run first to
    actually carry the last known level onto no-discovery days; cummax() after that is a cheap
    defensive guard in case any day's own value were ever lower than an earlier one."""
    path_data = data[(data['path_type'] == path_type) & (data['user_id'].isin(population))][['user_id', 'dt', 'max_item_level_reached']]
    path_data = path_data.groupby(['user_id', 'dt'], as_index=False)['max_item_level_reached'].max()

    grid = pd.MultiIndex.from_product([population, day_range], names=['user_id', 'dt']).to_frame(index=False)
    grid = grid.merge(path_data, on=['user_id', 'dt'], how='left').sort_values(['user_id', 'dt']).reset_index(drop=True)
    grid['current_level'] = grid.groupby('user_id')['max_item_level_reached'].transform(lambda s: s.ffill().cummax())
    grid['current_level'] = grid['current_level'].fillna(0).astype(int)
    return grid

level_grid_by_path = {
    path: build_current_level_grid(path, generator_population, day_range)
    for path in path_labels
}

In [13]:
for path, label in path_labels.items():
    grid = level_grid_by_path[path]
    max_level = grid['current_level'].max()

    # Funnel view: a player at level 5 counts toward every threshold 1-5, not just their own
    # level - so each line is its own cumulative "reached >= this level" reach curve (monotonic
    # non-decreasing over days), and at any given day the lines nest level-1-on-top-downward
    # (monotonic non-increasing across levels), unlike the old mutually-exclusive stacked
    # segments which summed to the population instead of nesting.
    funnel_rows = []
    for level in range(1, max_level + 1):
        daily = grid.groupby('dt')['current_level'].apply(lambda s, lv=level: (s >= lv).sum()).reset_index(name='n_users')
        daily['level'] = level
        funnel_rows.append(daily)
    funnel_counts = pd.concat(funnel_rows, ignore_index=True)
    funnel_counts['pct_of_population'] = (funnel_counts['n_users'] / n_population * 100).round(1)
    funnel_counts['level_label'] = 'Level ' + funnel_counts['level'].astype(str)
    level_order = [f'Level {i}' for i in range(1, max_level + 1)]

    fig = px.line(
        funnel_counts, x='dt', y='n_users', color='level_label',
        category_orders={'level_label': level_order},
        color_discrete_sequence=sequential_colors(max_level),
        markers=True,
        hover_data={'pct_of_population': ':.1f'},
        title=f'{label} Path: Reached >= Level, by Day (of {n_population:,}-player cohort)',
        labels={'dt': 'Date', 'n_users': 'Users', 'level_label': 'Level'},
        width=1200,
        height=500,
    )
    few_style(fig)
    fig.show()

In [14]:
for path, label in path_labels.items():
    grid = level_grid_by_path[path]

    pct_summary = grid.groupby('dt')['current_level'].agg(
        p10=lambda s: s.quantile(0.1),
        median=lambda s: s.quantile(0.5),
        p90=lambda s: s.quantile(0.9),
    ).reset_index()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=pd.concat([pct_summary['dt'], pct_summary['dt'][::-1]]),
        y=pd.concat([pct_summary['p90'], pct_summary['p10'][::-1]]),
        fill='toself',
        fillcolor='rgba(100,100,100,0.2)',
        line={'color': 'rgba(255,255,255,0)'},
        hoverinfo='skip',
        name='P10-P90',
    ))
    fig.add_trace(go.Scatter(
        x=pct_summary['dt'],
        y=pct_summary['median'],
        mode='lines+markers',
        name='Median',
        line={'color': _FEW_QUALITATIVE[0]},
    ))
    fig.update_layout(
        title=f'{label} Path: Median Level (P10-P90 band) Over Time - {n_population:,}-player cohort',
        xaxis_title='Date',
        yaxis_title='Current Level',
        width=1200,
        height=400,
    )
    few_style(fig)
    fig.show()

## Economy

Built on `aggregate_metric`/`aggregate_ratio_metric`/`plot_metric` - same CI bands, Test-vs-Control split, and Few-grounded light theme as `abtestmetrics`' own notebook - but copied into this project's own `common_lib/metrics.py` rather than imported across folders, so this notebook doesn't depend on a sibling project's file layout at runtime. Keep the two copies in sync by hand if the source changes; the local copy also adds the `test_end_date` marker used below.

**`TEST_END_DATE`** marks when the LiveOps event ends with a dotted vline on every chart, alongside the existing Test Start/Assignment Start markers.

**Filterable by generator engagement**: `GENERATOR_USE_THRESHOLD` below controls the Test population feeding every chart - `0` means every active Test player that day, no engagement filter; `N > 0` restricts Test to players with at least `N` generator activations *that specific day*. Control never has the generator, so it's always the full active control population regardless of threshold - each chart above 0 is really "heavy-engagement Test vs. the general control baseline," not a symmetric cut. Change the value and re-run this cell onward.

**Cumulative/overall-diff charts are intentionally disabled for every metric here** (`show_value_cumulative`/`show_rel_cumulative`/`show_overall_diff` all `False`) - those divide by `total_users_assigned`, a fixed cohort size, which stops meaning anything once `GENERATOR_USE_THRESHOLD > 0` makes the qualifying Test population change day by day. Only the daily value and daily relative-diff panels are shown.

`energy_earned`/`gems_earned` aren't plain columns in the live per-user pull - reconstructed using the exact dbt-verified formulas established earlier in this notebook (`energy_purchased + energy_earned_game + energy_earned_ads + energy_earned_gems`; `gems_purchased + gems_earned_game`), both confirmed to match the static `ab_dt_segment_metrics` table exactly. `energy_balance_end`/`gems_balance_end` come back from BigQuery as `Decimal` objects (BIGNUMERIC) rather than plain floats - cast before aggregating, or pandas' groupby/pivot machinery silently produces object-dtype columns downstream.


In [15]:
from common_lib.metrics import aggregate_metric, aggregate_ratio_metric, plot_metric

GENERATOR_USE_THRESHOLD = 0  # 0 = all active Test players that day; N>0 = Test players with >= N generator activations that specific day (control is always unfiltered)
TEST_END_DATE = '2026-08-31'  # LiveOps event end date - drawn as a vline on every chart below

ab_econ = ab_data_metrics[ab_data_metrics['variant'].isin(['Test', 'control'])].copy()
ab_econ['energy_earned'] = ab_econ['energy_purchased'] + ab_econ['energy_earned_game'] + ab_econ['energy_earned_ads'] + ab_econ['energy_earned_gems']
ab_econ['gems_earned'] = ab_econ['gems_purchased'] + ab_econ['gems_earned_game']
ab_econ['energy_balance_end'] = ab_econ['energy_balance_end'].astype(float)
ab_econ['gems_balance_end'] = ab_econ['gems_balance_end'].astype(float)

if GENERATOR_USE_THRESHOLD == 0:
    filtered_econ = ab_econ
else:
    eligible = generator_activated_daily[generator_activated_daily['n_activations'] >= GENERATOR_USE_THRESHOLD][['user_id', 'dt']]
    test_filtered = ab_econ[ab_econ['variant'] == 'Test'].merge(eligible, on=['user_id', 'dt'], how='inner')
    control_unfiltered = ab_econ[ab_econ['variant'] == 'control']
    filtered_econ = pd.concat([test_filtered, control_unfiltered], ignore_index=True)

ECONOMY_PLOT_KWARGS = dict(
    show_overall_diff=False, show_value=True, show_rel=True,
    show_value_cumulative=False, show_rel_cumulative=False,
    assignment_start_date='2026-08-05', test_start_date='2026-08-17', test_end_date=TEST_END_DATE,
    width=1500, height=400,
)

def plot_economy_currency(currency: str, spent: str, earned: str, balance: str, velocity: str, population: pd.DataFrame) -> None:
    for metric in [spent, earned, balance]:
        agg = aggregate_metric(population, metric, 'mean', ab_assign)
        plot_metric(agg, metric, **ECONOMY_PLOT_KWARGS)
    velocity_agg = aggregate_ratio_metric(population, spent, earned, velocity)
    plot_metric(velocity_agg, velocity, **ECONOMY_PLOT_KWARGS)


### Energy

In [16]:
plot_economy_currency('energy', 'energy_spent', 'energy_earned', 'energy_balance_end', 'energy_velocity', filtered_econ)

### Gems

In [17]:
plot_economy_currency('gems', 'gems_spent', 'gems_earned', 'gems_balance_end', 'gems_velocity', filtered_econ)

### Overflow Farmers vs Control (exploratory - heavily biased population)

**Step 1 of a two-step idea, park the bias for now.** Redefines "Test" for this subsection as only the players who ever completed a milestone beyond 16 (the farmable overflow zone - see [[Milestones]] below) - compared against the same full, unfiltered control population used everywhere else in this notebook.

**This population is not a random Test subset - it's heavily biased by construction.** Reaching milestone 17+ requires having already cleared the entire 1-16 ladder, so "overflow reachers" are by definition the most active, highest-engagement players in Test. Any economy gap shown here conflates "the farming behavior itself" with "just being an unusually engaged player" - it cannot separate the two. That's exactly why this is step 1, not a conclusion: a real read on farming's own effect would need to match these players against comparably-engaged control players first (step 2, not attempted here).


In [ ]:
# Reload milestone_timing here (Economy now runs before Milestones - see [[Milestones]] below) -
# load_or_query just reads the existing pickle when refresh_data=False, so this is effectively
# free, not a second real pull.
milestone_timing_for_farmers = bqc.load_or_query('milestone_timing', refresh_data)

overflow_user_ids = milestone_timing_for_farmers[milestone_timing_for_farmers['milestone'] > 16]['user_id'].unique()
print(f'{len(overflow_user_ids):,} players ever reached a milestone beyond 16')

farmer_test = ab_econ[(ab_econ['variant'] == 'Test') & (ab_econ['user_id'].isin(overflow_user_ids))]
control_unfiltered = ab_econ[ab_econ['variant'] == 'control']
farmer_econ = pd.concat([farmer_test, control_unfiltered], ignore_index=True)


#### Energy

In [ ]:
plot_economy_currency('energy', 'energy_spent', 'energy_earned', 'energy_balance_end', 'energy_velocity', farmer_econ)

#### Gems

In [ ]:
plot_economy_currency('gems', 'gems_spent', 'gems_earned', 'gems_balance_end', 'gems_velocity', farmer_econ)

## Milestones

How far players got, how much it cost them (in event points), and how fast they cleared each stage - covering both the designed 1-16 ladder and the farmable >16 overflow zone.

### Time to Complete

Time between consecutive milestone completions for the point-milestone ladder (`context IN ('goal','goalCompleted')`, scoped to the reward-eligible range - event milestones 1-16 (reward stages 0-15). Milestone 1's time is measured from the event's own start (`liveops_event_start_ts`), not from an earlier reference. See [sql/milestone_timing.sql](./sql/milestone_timing.sql) - it pulls the full milestone range (no upper bound), not just 1-16, so the same pull can support the Q3 >16 investigation later without a second query.

**Caveat:** 7% of users skip at least one milestone number in their completion sequence (found while planning this). A delta attributed to milestone N is computed against each user's previous *observed* milestone, not a literal N-1 - so for skippers, that delta actually spans multiple stages, not one. The table below reports `pct_single_step` per milestone so this is visible rather than silently absorbed into the average.

A handful of deltas (11 out of 252k rows, 5 of them in the 1-16 range) came out negative - near-simultaneous or out-of-order completion events, not a real negative duration. Clipped to 0 rather than dropped, since the milestone genuinely was completed. One of the two users behind these is also in the >16 overflow cohort, with milestone numbers arriving out of chronological order (not just fast) - a new data point for the Q3 investigation, not resolved here.

In [18]:
# hide-output 
milestone_timing = bqc.load_or_query('milestone_timing', refresh_data)

This query will process 1.47 GB when run.
Estimated query cost: $0.01


In [19]:
# hide-output 
#milestone_timing

In [20]:
# hide-output 
# format='ISO8601' - BigQuery's DATETIME values come back from bqc.get() with inconsistent
# fractional-second precision (some rows have microseconds, some don't, e.g. "...21:40:40" vs
# "...21:40:40.377499"). A bare pd.to_datetime() infers one fixed format from the first rows and
# then throws on the first row that doesn't match it - format='ISO8601' parses each value on its
# own terms instead of assuming one shared format.
milestone_timing['completion_ts'] = pd.to_datetime(milestone_timing['completion_ts'], format='ISO8601')
milestone_timing['liveops_event_start_ts'] = pd.to_datetime(milestone_timing['liveops_event_start_ts'], format='ISO8601')

milestone_timing = milestone_timing.sort_values(['user_id', 'milestone']).reset_index(drop=True)

milestone_timing['prev_completion_ts'] = milestone_timing.groupby('user_id')['completion_ts'].shift(1)
milestone_timing['prev_milestone'] = milestone_timing.groupby('user_id')['milestone'].shift(1)

# A user's first observed milestone has no prior milestone to diff against - use the event's own
# start as the baseline, so "time to complete milestone 1" means time since the event went live.
milestone_timing['prev_completion_ts'] = milestone_timing['prev_completion_ts'].fillna(milestone_timing['liveops_event_start_ts'])

milestone_timing['time_to_complete_hours'] = (
    milestone_timing['completion_ts'] - milestone_timing['prev_completion_ts']
).dt.total_seconds() / 3600

# Near-simultaneous or out-of-order completion events produce a handful of negative deltas (11 of
# 252k rows) - clipped to 0 rather than dropped, since the milestone genuinely was completed and
# dropping the row would silently shrink that milestone's n_users.
milestone_timing['time_to_complete_hours'] = milestone_timing['time_to_complete_hours'].clip(lower=0)

# ~7% of users skip at least one milestone number - a delta attributed to milestone N where
# prev_milestone != N-1 actually spans multiple stages, not one. Flagged (not dropped) so the
# caveat is visible in the aggregate rather than silently absorbed into it.
milestone_timing['is_single_step'] = (
    milestone_timing['prev_milestone'].isna() | (milestone_timing['milestone'] - milestone_timing['prev_milestone'] == 1)
)

In [21]:
reward_eligible_timing = milestone_timing[milestone_timing['milestone'].between(1, 16)]

milestone_timing_agg = reward_eligible_timing.groupby('milestone').agg(
    n_users=('user_id', 'nunique'),
    mean_hours=('time_to_complete_hours', 'mean'),
    median_hours=('time_to_complete_hours', 'median'),
    n_single_step=('is_single_step', 'sum'),
).reset_index()

milestone_timing_agg['pct_single_step'] = (milestone_timing_agg['n_single_step'] / milestone_timing_agg['n_users'] * 100).round(1)
milestone_timing_agg[['mean_hours', 'median_hours']] = milestone_timing_agg[['mean_hours', 'median_hours']].round(2)
milestone_timing_agg = milestone_timing_agg.drop(columns=['n_single_step'])

#milestone_timing_agg

In [22]:
mean_median_long = milestone_timing_agg.melt(
    id_vars='milestone', value_vars=['mean_hours', 'median_hours'],
    var_name='stat', value_name='hours',
)
mean_median_long['stat'] = mean_median_long['stat'].map({'mean_hours': 'Mean', 'median_hours': 'Median'})

fig = px.line(
    mean_median_long, x='milestone', y='hours', color='stat',
    color_discrete_map={'Mean': _FEW_QUALITATIVE[0], 'Median': _FEW_QUALITATIVE[1]},
    markers=True,
    title='Time to Complete Each Milestone (hours) - Mean vs Median',
    labels={'milestone': 'Milestone', 'hours': 'Hours', 'stat': ''},
    width=1200,
    height=500,
)
few_style(fig)
fig.show()

### Points

`GeneratorEvent-GardenGlory` tracks a real, event-specific currency called `generatorEventPoints` (found via `dim_liveops.liveops_currency_id_array`/`liveops_currency_tracking_name_array`), a persistent, never-spent running balance (`outflow` is always 0). It's tracked at exact-timestamp grain in `rp_ts_user_events_liveops_economy` - the same raw source `fact_dt_user_liveops_economy` itself is built from (confirmed by reading the dbt model directly), not just the daily-grain fact table.

See [sql/points_at_milestone.sql](./sql/points_at_milestone.sql) - an as-of join (carry-forward last known balance via `LAST_VALUE(... IGNORE NULLS)` over the merged, time-sorted stream of points-events and milestone-completions) gives the exact points balance at or immediately before each milestone completion, not a daily approximation.

**Caveat on the low end:** milestones 1-4 have low match rates (16-33%, `match_rate_pct` column) - most users cross those very early thresholds before their *first* recorded points-event ever fires (points appear to log at a coarser cadence than instant-per-action), so those numbers come from a smaller, possibly-biased subsample. Match rate is 88%+ by milestone 7 and 97%+ by milestone 8.

In [23]:
# hide-output 
points_at_milestone = bqc.load_or_query('points_at_milestone', refresh_data)

This query will process 3.86 GB when run.
Estimated query cost: $0.03


In [24]:
# hide-output 
#points_at_milestone

In [25]:
points_milestone_agg = points_at_milestone.groupby('milestone').agg(
    n_users=('user_id', 'count'),
    n_no_prior_points_event=('points_balance_at_or_before', lambda s: s.isna().sum()),
    median_points=('points_balance_at_or_before', 'median'),
    p25_points=('points_balance_at_or_before', lambda s: s.quantile(0.25)),
    p75_points=('points_balance_at_or_before', lambda s: s.quantile(0.75)),
).reset_index()

points_milestone_agg['match_rate_pct'] = (100 - points_milestone_agg['n_no_prior_points_event'] / points_milestone_agg['n_users'] * 100).round(1)
points_milestone_agg['median_points'] = points_milestone_agg['median_points'].round(0)
# Points needed to clear this milestone specifically, i.e. the marginal cost - not the
# cumulative total, which is what median_points itself already is.
points_milestone_agg['delta_from_prev'] = points_milestone_agg['median_points'].diff()
points_milestone_agg = points_milestone_agg.drop(columns=['n_no_prior_points_event'])

ladder_points_table = points_milestone_agg[points_milestone_agg['milestone'].between(1, 16)].reset_index(drop=True)
overflow_points_table = points_milestone_agg[points_milestone_agg['milestone'] > 16].reset_index(drop=True)

In [26]:
# Chart-only filter (n_users >= 100) - same convention as the velocity charts. Only matters for
# the overflow table in practice (it thins out to single-digit users by milestone ~35), the
# ladder table never drops below ~3,600 users so this is a no-op there.
ladder_points_chart_data = ladder_points_table[ladder_points_table['n_users'] >= 100].copy()
ladder_points_chart_data['series'] = '1-16 (designed ladder)'
overflow_points_chart_data = overflow_points_table[overflow_points_table['n_users'] >= 100].copy()
overflow_points_chart_data['series'] = '17+ (overflow)'
points_chart_data = pd.concat([ladder_points_chart_data, overflow_points_chart_data], ignore_index=True)
series_order = ['1-16 (designed ladder)', '17+ (overflow)']

fig = px.line(
    points_chart_data, x='milestone', y='median_points', color='series',
    category_orders={'series': series_order},
    color_discrete_map=_LADDER_COLORS,
    markers=True,
    hover_data={'n_users': ':,'},
    title='Points Required per Milestone (median) - escalating curve vs flat overflow rate (n_users >= 100)',
    labels={'milestone': 'Milestone', 'median_points': 'Cumulative Points', 'series': ''},
    width=1200,
    height=500,
)
few_style(fig)
fig.show()

#### Marginal cost per milestone

The *marginal* cost (`delta_from_prev`, points needed for that specific milestone, not the cumulative total) makes the farming pattern obvious: an escalating climb through milestone 16 (93 points for milestone 2 up to 1,560 for milestone 16 - a deliberately increasing difficulty curve), then an abrupt reset to a **flat ~600 points per milestone** (mean 600.1, std 72.1 across all 146 overflow milestones) that never resumes escalating.

That flat rate is roughly the milestone 5-6 difficulty level - cheap for a player who has already built up the point-earning rate needed to clear the whole escalating ladder to get here. A real, quantitative confirmation of the farming suspicion: the design's escalating difficulty is bypassed entirely past 16, not just "uncapped" as LPD-261 originally concluded.

In [27]:
fig = px.line(
    points_chart_data, x='milestone', y='delta_from_prev', color='series',
    category_orders={'series': series_order},
    color_discrete_map=_LADDER_COLORS,
    markers=True,
    hover_data={'n_users': ':,'},
    title='Marginal Points Cost per Milestone - escalating then flat (n_users >= 100)',
    labels={'milestone': 'Milestone', 'delta_from_prev': 'Points needed for this milestone', 'series': ''},
    width=1200,
    height=500,
)
few_style(fig)
fig.show()

### Reward Claims

Same funnel pattern as the charts above, but built from reward **claim** dates (`event_dt`) rather than milestone **completion** dates - a player at stage 5 counts toward every line from stage 0 through 5. Pulls `reward_tracks` (see [sql/reward_tracks.sql](./sql/reward_tracks.sql) - full breakdown of all 9 reward tracks in the Reward Tracks section below), scoped here to just the ladder-specific tracks (`ladder_0_15` + `ladder_16_plus`), which are one continuous numeric progression; stages 16+ are aggregated into a single "16+ (overflow)" line for the same readability reason as the milestone chart.

Sanity check: this chart's stage 0-15 last-day counts (25,534 -> 3,255) match the equivalent numbers independently derived from `rp_LOpsMilestoneComplete` earlier in this notebook, confirming claim-based and completion-based counts agree at the point-milestone grain.

In [28]:
# hide-output 
reward_tracks = bqc.load_or_query('reward_tracks', refresh_data)

# Fixes a labeling bug from this session's first pull: the path-completion bonus regex only
# captured the 'pathN' group, merging path0_complete/path1_complete/path2_complete into their
# parent path0/path1/path2 track. is_completion_bonus survived correctly though, so this is safe
# to fix here - and idempotent against a fresh pull already using the corrected SQL (a reward_track
# already ending in '_complete' is left untouched, not double-suffixed).
_needs_fix = (
    reward_tracks['reward_track'].str.startswith('path')
    & reward_tracks['is_completion_bonus']
    & ~reward_tracks['reward_track'].str.endswith('_complete')
)
reward_tracks.loc[_needs_fix, 'reward_track'] = reward_tracks.loc[_needs_fix, 'reward_track'] + '_complete'

This query will process 8.64 GB when run.
Estimated query cost: $0.06


In [29]:
# hide-output 
#reward_tracks

In [30]:
reward_tracks['stage'] = pd.to_numeric(reward_tracks['reward_suffix'], errors='coerce')

ladder_claims = reward_tracks.dropna(subset=['stage'])[['user_id', 'stage', 'event_dt']].copy()
ladder_claims['event_dt'] = pd.to_datetime(ladder_claims['event_dt'])
ladder_claims['stage'] = ladder_claims['stage'].astype(int)

reward_funnel_rows = []
for stage in range(0, 16):
    dates = ladder_claims[ladder_claims['stage'] == stage].groupby('user_id')['event_dt'].min()
    counts = [(dates <= day).sum() for day in day_range]
    df = pd.DataFrame({'dt': day_range, 'n_users': counts})
    df['stage_label'] = f'Stage {stage}'
    reward_funnel_rows.append(df)

# 16+: first time a user claimed ANY stage >= 16 - the overflow-zone equivalent of a single
# threshold, same as the milestone chart's "17+ (overflow)" line.
dates_16plus = ladder_claims[ladder_claims['stage'] >= 16].groupby('user_id')['event_dt'].min()
counts_16plus = [(dates_16plus <= day).sum() for day in day_range]
df_16plus = pd.DataFrame({'dt': day_range, 'n_users': counts_16plus})
df_16plus['stage_label'] = '16+ (overflow)'
reward_funnel_rows.append(df_16plus)

reward_funnel_counts = pd.concat(reward_funnel_rows, ignore_index=True)

reward_labels_order = [f'Stage {i}' for i in range(16)] + ['16+ (overflow)']
# Sequential blues for the 16 designed stages (ordered), then the same warm overflow accent
# used everywhere else in the notebook for the 17+ farming zone - reserves the one saturated
# color for what should draw the eye, per Few.
reward_stage_colors = dict(zip([f'Stage {i}' for i in range(16)], sequential_colors(16)))
reward_stage_colors['16+ (overflow)'] = '#C87941'

fig = px.line(
    reward_funnel_counts, x='dt', y='n_users', color='stage_label',
    category_orders={'stage_label': reward_labels_order},
    color_discrete_map=reward_stage_colors,
    markers=True,
    title='Reward Ladder Claims (>= stage), by Day',
    labels={'dt': 'Date', 'n_users': 'Users', 'stage_label': 'Stage'},
    width=1200,
    height=500,
)
few_style(fig)
#fig.show()

### Velocity

If the overflow zone is genuinely farmable (cheap, flat points cost), players should also be *clearing* those milestones faster, not just paying less for them. Reuses `milestone_timing` from Q2 - it already covers the full milestone range, not just 1-16, since it was built for exactly this kind of follow-up. Velocity is `1 / median_hours` (milestones per hour); a median of exactly 0 hours (the same rare same-instant multi-milestone completions found in Q2 - batch-processed, not really instantaneous) is treated as missing rather than an infinite velocity.

**Result: speed does increase, and keeps increasing.** Median velocity across the ladder's last few milestones (12-16) is ~0.029/hour; across the first stretch of overflow (17-30) it's ~0.090/hour - about 3x faster on average. It's not a one-time reset either: velocity keeps climbing through the overflow zone itself (0.071 at milestone 17 up to 0.33 by milestone 36), suggesting whatever is driving this (repeatable/automatable action, or just very high-throughput players) keeps getting more efficient the further in it goes - not just a flat "somewhat faster than before."

In [31]:
# hide-output 
velocity_agg = milestone_timing.groupby('milestone').agg(
    n_users=('user_id', 'nunique'),
    median_hours=('time_to_complete_hours', 'median'),
    n_single_step=('is_single_step', 'sum'),
).reset_index()

velocity_agg['pct_single_step'] = (velocity_agg['n_single_step'] / velocity_agg['n_users'] * 100).round(1)
velocity_agg = velocity_agg.drop(columns=['n_single_step'])

# A median of exactly 0 hours (rare same-instant multi-milestone completions) would give an
# infinite velocity - treated as missing rather than plotted/reported as inf.
velocity_agg['velocity_per_hour'] = np.where(velocity_agg['median_hours'] > 0, 1 / velocity_agg['median_hours'], np.nan)
velocity_agg[['median_hours', 'velocity_per_hour']] = velocity_agg[['median_hours', 'velocity_per_hour']].round(4)

ladder_velocity_table = velocity_agg[velocity_agg['milestone'].between(1, 16)].reset_index(drop=True)
overflow_velocity_table = velocity_agg[velocity_agg['milestone'] > 16].reset_index(drop=True)

In [32]:
# Chart-only filtering (the tables above keep full detail) - milestones 1-2 are near-instant
# for most players (milestone 2's median was 0.37 hours) and read as automatic/bundled with
# generator activation rather than a real pacing signal, and the thin tail (n_users <= 100,
# milestone ~31+) is single-/few-user noise that dominates the y-axis and hides the actual curve.
ladder_velocity_chart_data = ladder_velocity_table[(ladder_velocity_table['milestone'] > 2) & (ladder_velocity_table['n_users'] > 100)].copy()
ladder_velocity_chart_data['series'] = '1-16 (designed ladder)'
overflow_velocity_chart_data = overflow_velocity_table[(overflow_velocity_table['milestone'] > 2) & (overflow_velocity_table['n_users'] > 100)].copy()
overflow_velocity_chart_data['series'] = '17+ (overflow)'
velocity_chart_data = pd.concat([ladder_velocity_chart_data, overflow_velocity_chart_data], ignore_index=True)

fig = px.line(
    velocity_chart_data, x='milestone', y='velocity_per_hour', color='series',
    category_orders={'series': series_order},
    color_discrete_map=_LADDER_COLORS,
    markers=True,
    hover_data={'n_users': ':,'},
    title='Milestone Claim Velocity - speed increases past the designed ladder (milestone 3+, n_users > 100)',
    labels={'milestone': 'Milestone', 'velocity_per_hour': 'Velocity (milestones per hour)', 'series': ''},
    width=1200,
    height=500,
)
few_style(fig)
fig.show()

#### Alternative view: cumulative pace (milestones/day since event start)

The chart above measures the *marginal* pace - time between two specific adjacent milestones - which is why milestones 1-2 looked "automatic" (a short, noisy local gap) and why the thin tail was so volatile. This alternative instead measures **cumulative throughput**: `milestone number / days elapsed since the event started` - i.e. "by the time this player reached milestone N, how many milestones per day had they averaged overall." Dividing by elapsed days directly answers the "how long have they been playing" question, rather than leaving it as an unknown baked into a per-hour marginal rate.

Same story, more intuitive units and a smoother curve: pace starts high during the initial engagement burst (milestones 1-4, ~2-3/day), drops as the ladder gets harder (~1.2/day around milestone 8), then climbs back up through the rest of the ladder and *keeps climbing* into the overflow zone (1.41/day at milestone 16 -> 2.37/day by milestone 30) - the same acceleration the marginal-velocity chart showed, without needing to drop milestones 1-2 to see it.

In [33]:
milestone_timing['days_elapsed'] = (milestone_timing['completion_ts'] - milestone_timing['liveops_event_start_ts']).dt.total_seconds() / 86400
milestone_timing['cumulative_milestones_per_day'] = milestone_timing['milestone'] / milestone_timing['days_elapsed']

pace_agg = milestone_timing.groupby('milestone').agg(
    n_users=('user_id', 'nunique'),
    median_pace=('cumulative_milestones_per_day', 'median'),
).reset_index()
pace_agg['median_pace'] = pace_agg['median_pace'].round(4)

ladder_pace_table = pace_agg[pace_agg['milestone'].between(1, 16)].reset_index(drop=True)
overflow_pace_table = pace_agg[pace_agg['milestone'] > 16].reset_index(drop=True)

# Same n_users > 100 chart-only filter as the marginal-velocity chart, for consistency - this
# metric doesn't need the milestone > 2 filter too, since it's not distorted by a short noisy
# local gap the way the marginal per-hour rate was.
ladder_pace_chart_data = ladder_pace_table[ladder_pace_table['n_users'] > 100].copy()
ladder_pace_chart_data['series'] = '1-16 (designed ladder)'
overflow_pace_chart_data = overflow_pace_table[overflow_pace_table['n_users'] > 100].copy()
overflow_pace_chart_data['series'] = '17+ (overflow)'
pace_chart_data = pd.concat([ladder_pace_chart_data, overflow_pace_chart_data], ignore_index=True)

In [34]:
fig = px.line(
    pace_chart_data, x='milestone', y='median_pace', color='series',
    category_orders={'series': series_order},
    color_discrete_map=_LADDER_COLORS,
    markers=True,
    hover_data={'n_users': ':,'},
    title='Cumulative Milestone Pace (milestones/day since event start, n_users > 100)',
    labels={'milestone': 'Milestone', 'median_pace': 'Milestones per day (cumulative average)', 'series': ''},
    width=1200,
    height=500,
)
few_style(fig)
fig.show()

#### Third view: velocity by calendar day, not by milestone number

Both charts above put milestone number on the x-axis - useful for "how does difficulty/pace change as you go deeper," but they can't show *when in the event's calendar* the overflow-driven speedup actually shows up, which is what matters for lining this up against other date-anchored observations (like the Aug 25 monetisation flip). This one puts calendar date on the x-axis instead: for each day, the median velocity (in milestones/day, `24 / median_hours`) among Ladder (1-16) completions that day vs Overflow (17+) completions that day.

**Chart-only filter**: Overflow's first few days (Aug 18-20) have only 14-30 completions total and show wild, small-sample velocities (60-100+/day) that would dominate the y-axis - dropped via `n_completions >= 50`. Once the overflow population is large enough to be stable (Aug 22 on), its velocity holds well above Ladder's for the rest of the event, and the gap widens as Ladder's own pace keeps declining (survivorship - the players still working through the ladder late are increasingly the slower ones). The dashed line marks Aug 25.

In [35]:
milestone_timing['completion_dt'] = milestone_timing['completion_ts'].dt.normalize()
milestone_timing['ladder_group'] = np.where(milestone_timing['milestone'] <= 16, 'Ladder (1-16)', 'Overflow (17+)')

daily_velocity = milestone_timing.groupby(['completion_dt', 'ladder_group']).agg(
    n_completions=('user_id', 'count'),
    median_hours=('time_to_complete_hours', 'median'),
).reset_index()
daily_velocity['velocity_per_day'] = np.where(daily_velocity['median_hours'] > 0, 24 / daily_velocity['median_hours'], np.nan)

# Drop the small-sample early-overflow spike (Aug 18-20, n_completions in the teens/twenties) -
# chart-only, same convention as the other two velocity charts.
daily_velocity_chart_data = daily_velocity[
    (daily_velocity['completion_dt'] >= pd.Timestamp('2026-08-17'))
    & (daily_velocity['completion_dt'] <= pd.Timestamp('2026-08-31'))
    & (daily_velocity['n_completions'] >= 50)
]

In [36]:
fig = px.line(
    daily_velocity_chart_data, x='completion_dt', y='velocity_per_day', color='ladder_group',
    category_orders={'ladder_group': ['Ladder (1-16)', 'Overflow (17+)']},
    color_discrete_map=_LADDER_COLORS,
    markers=True,
    hover_data={'n_completions': ':,'},
    title='Milestone Velocity by Calendar Day - Ladder vs Overflow (n_completions >= 50)',
    labels={'completion_dt': 'Date', 'velocity_per_day': 'Velocity (milestones per day)', 'ladder_group': ''},
    width=1200,
    height=500,
)
fig.add_vline(x=pd.Timestamp('2026-08-25').timestamp() * 1000, line_dash='dash', line_color='gray', annotation_text='Aug 25')
few_style(fig)
fig.show()

# APPENDIX

## APPENDIX Reward Tracks (corrected) - feeds Q3 & Q4

The original `rewards.sql` query only ever captured one of at least **9 real reward tracks** tied to this event, because it filters `wallet_diff_array` to `id LIKE '%liveops-GeneratorEvent-GardenGlory%'` - correct for isolating the event's own dedicated currency, but it silently drops everything else (that narrower query and its one-track view have since been removed as redundant - the `ladder_0_15` track already shown earlier, in the Ladder Milestones section's Reward Claims subsection, reproduces it exactly, day-by-day). Found by inspecting the raw `reward_id` space directly (see [sql/reward_tracks.sql](./sql/reward_tracks.sql)):

- `ladder_0_15` - the point-milestone ladder's own dedicated currency.
- `ladder_16_plus` - **point-milestone completions past 15 still trigger real reward claims** (confirmed: real `reward_instance_id`s, real users), just granting generic/cross-promo items (other campaigns' chests, plain energy) instead of this event's own currency. This directly updates LPD-261: overflow milestones are not unrewarded, as previously concluded - they're rewarded from an unrelated item pool once GardenGlory's own 16-slot reward table is exhausted.
- `path0` / `path1` / `path2` - per-level rewards for each item-path (generator/primary/secondary), granting generic currencies (gems/energy) and cross-promo chests. Never GardenGlory-branded, so never captured by the old narrower query at all.
- `path0_complete` / `path1_complete` / `path2_complete` - one-time bonus for fully finishing a given item-path.
- `complete` - one-time bonus for finishing the full 16-milestone ladder.

**Data-quality notes from building this:**
- `dynamic_rewards_item_array` is always empty for this event - every reward flows through `wallet_diff_array`, checked directly.
- `reward_instance_id` is *not* a stable one-row-per-claim key here - it's shared across many distinct `reward_id`s for the same user (a claim-batch/transaction id, not a per-reward id). `(user_id, reward_id)` is the real claim grain.
- A small number of users (16 out of ~26,400) have genuine repeat claims of the identical `reward_id` (up to 3x) - deduplicated via `ROW_NUMBER()` in the SQL, keeping the first instance.
- Checked separately: the feature stopped producing new milestone completions essentially at the event's actual end (`liveops_event_end_ts` = 2026-08-31 10:00 UTC) - Aug 31 has a partial day of legitimate tail activity, Sep 1 has next to nothing (5 completions from 3 users) - so this data isn't still shifting under us.

In [37]:
# (user_id, reward_id) is the real claim grain - reward_tracks has one row per wallet item per
# claim (a single claim can grant more than one wallet_diff_array entry), so drop back to the
# claim grain before counting claims/users per track.
claims_by_track = reward_tracks[['user_id', 'reward_id', 'reward_track']].drop_duplicates()

reward_track_summary = claims_by_track.groupby('reward_track').agg(
    n_claims=('reward_id', 'count'),
    n_users=('user_id', 'nunique'),
).reset_index().sort_values('n_claims', ascending=False)

reward_track_summary

,reward_track,n_claims,n_users
1,ladder_0_15,237321,25626
5,path1,229468,25528
3,path0,125455,26251
7,path2,65999,13458
2,ladder_16_plus,16617,3122
6,path1_complete,13274,13274
8,path2_complete,12406,12406
4,path0_complete,3642,3642
0,complete,3633,3633


In [38]:
fig = px.bar(
    reward_track_summary,
    x='reward_track',
    y='n_users',
    title='Reward Tracks: Distinct Users per Track (corrected, all 9 tracks)',
    width=1200,
    height=500,
    labels={'reward_track': 'Reward Track', 'n_users': 'Users'},
    hover_data={'n_claims': True},
    category_orders={'reward_track': reward_track_summary.sort_values('n_users', ascending=False)['reward_track'].tolist()},
    color_discrete_sequence=[_FEW_QUALITATIVE[0]],
)
few_style(fig)
fig.show()

In [39]:
# What each track actually grants - this is what shows ladder_16_plus/path0/path1/path2 handing
# out generic/cross-promo items (gems, energy, other campaigns' chests) rather than this event's
# own currency, unlike ladder_0_15's dedicated 'liveops-GeneratorEvent-GardenGlory-Gen-level N'.
top_items_by_track = (
    reward_tracks.dropna(subset=['wallet_id'])
    .groupby(['reward_track', 'wallet_id'])
    .size()
    .reset_index(name='n_rows')
    .sort_values(['reward_track', 'n_rows'], ascending=[True, False])
    .groupby('reward_track')
    .head(3)
)

top_items_by_track

,reward_track,wallet_id,n_rows
0,complete,DreamSticker,3633
1,complete,Playtime-Pack_5_L,3633
2,ladder_0_15,liveops-GeneratorEvent-GardenGlory-Gen-level 1,97414
3,ladder_0_15,liveops-GeneratorEvent-GardenGlory-Gen-level 2,69994
4,ladder_0_15,liveops-GeneratorEvent-GardenGlory-Gen-level 3,45858
13,ladder_16_plus,mergePath-picnicPassChest-Level 1,3477
12,ladder_16_plus,mergePath-herbs-level 1,3474
7,ladder_16_plus,Playtime-Pack_4_L,2409
15,path0,gems,125455
16,path0_complete,Playtime-Pack_4_L,3642


### Full reward catalog - what each reward actually grants

Every distinct reward and its item/currency, reconstructed directly from claim data. Verified first that this is safe to do: for every one of the 188 distinct `reward_id`s, the exact set of items granted is identical across every claim of it (checked directly, 0 exceptions) - so a reward's configuration can be read straight off any of its claims rather than needing a separate config table.

**Notable pattern found here**: `ladder_16_plus` isn't random filler - it's a **repeating 6-item cycle** (herbs -> PicnicPass chest -> Playtime pack -> energy currency -> events currency chest -> 100 energy -> repeats) that continues unchanged all the way to stage 161. Once GardenGlory's own 16-slot reward table is exhausted, the game falls through to a fixed generic loop rather than stopping or scaling with depth.

In [40]:
reward_catalog = reward_tracks.dropna(subset=['wallet_id'])[['reward_track', 'reward_suffix', 'stage', 'wallet_id', 'wallet_count']].drop_duplicates()

# Item number for path-track rewards (e.g. 'path0_item2' -> 2), for sort/display ordering -
# stage already covers the ladder tracks.
reward_catalog['item_number'] = reward_catalog['reward_suffix'].str.extract(r'_item(\d+)$').astype(float)

def detail_label(row):
    if pd.notna(row['stage']):
        return f'Stage {int(row["stage"])}'
    if pd.notna(row['item_number']):
        return f'Item {int(row["item_number"])}'
    return 'Completion Bonus'

reward_catalog['detail'] = reward_catalog.apply(detail_label, axis=1)
reward_catalog = reward_catalog.sort_values(['reward_track', 'stage', 'item_number', 'wallet_id'])
reward_catalog = reward_catalog[['reward_track', 'detail', 'wallet_id', 'wallet_count']].reset_index(drop=True)

reward_catalog

,reward_track,detail,wallet_id,wallet_count
0,complete,Completion Bonus,DreamSticker,1
1,complete,Completion Bonus,Playtime-Pack_5_L,1
2,ladder_0_15,Stage 0,liveops-GeneratorEvent-GardenGlory-Gen-level 1,1
3,ladder_0_15,Stage 1,liveops-GeneratorEvent-GardenGlory-Gen-level 1,1
4,ladder_0_15,Stage 2,liveops-GeneratorEvent-GardenGlory-Gen-level 1,1
...,...,...,...,...
187,path2,Item 2,energy,1
188,path2,Item 3,energy,1
189,path2,Item 4,energy,1
190,path2_complete,Completion Bonus,Playtime-Pack_3_L,1


## APPENDIX Max item reach by day

In [41]:
# hide-output 
milestone_daily_agg = data.groupby(['dt','path_tier','path_type','max_item_level_reached']).agg(
    users = ('user_id','nunique'),
    n_levels_discovered_avg = ('n_levels_discovered','mean'),
    n_levels_discovered_median = ('n_levels_discovered','median'),
    #max_item_level_reached = ('max_item_level_reached','max'),
).reset_index()

#milestone_daily_agg


In [42]:
# hide-output 
_n_dt = milestone_daily_agg['dt'].nunique()
fig = px.line(milestone_daily_agg, 
              x='max_item_level_reached', 
              y='users',
              color='dt',
              color_discrete_sequence=sequential_colors(_n_dt),
              title='Distribution of max item level reached by day: All Paths',
              width=1200,
              height=1000,
              facet_row='path_type',
              hover_data={'users': True, 'n_levels_discovered_avg': True, 'n_levels_discovered_median': True, 'max_item_level_reached': True},
              )

few_style(fig)
#fig.show()


In [43]:
# hide-output 
milestone_agg = data.groupby(['dt','path_tier','path_type','max_item_level_reached']).agg(
    users = ('user_id','nunique'),
    n_levels_discovered_avg = ('n_levels_discovered','mean'),
    n_levels_discovered_median = ('n_levels_discovered','median'),
    #max_item_level_reached = ('max_item_level_reached','max'),
).reset_index()

#milestone_agg

## APPENDIX Milestone Distribution (rp_LOpsMilestoneComplete reproduction)

Independent reproduction of Omni's "Milestone Distribution" tile, built directly from `rp_LOpsMilestoneComplete` with `context IN ('goal', 'goalCompleted')` - explicitly excluding `itemPath` rows (see [sql/milestones.sql](./sql/milestones.sql)). Background: Omni's `fact_dt_user_liveops_ms_event_progression` has no `context` column at all and shows per-player furthest milestones running up to 112 against this event's configured 16-stage ladder ([LPD-261](https://trailmixgames.atlassian.net/browse/LPD-261)). If this reproduction caps cleanly at 16, that's strong evidence the >16 tail in Omni comes from a missing `context` filter upstream in the dbt model that materializes that fact, not from the raw event data itself.

The query also joins in the full event funnel population (`fact_dt_user_liveops_event_funnel`, left-joined the same way Omni's topic joins it - `always_left` on user_id + event definition + iteration), so participants who never completed a milestone still appear with a NULL furthest milestone. That reproduces Omni's blank "0 milestones" bucket rather than silently excluding those players.

In [44]:
# hide-output 
milestones = bqc.load_or_query('milestones', refresh_data)

This query will process 2.87 GB when run.
Estimated query cost: $0.02


In [45]:
# hide-output 
#milestones

In [46]:
# hide-output 
# Collapse to one row per player (furthest milestone reached across the whole iteration) -
# grouping on the daily max_milestone_completed directly would over-count players who
# completed milestones across multiple days, same trap Omni's ai_context warns about.
# Players with no completions have a single all-NULL row from the funnel left join, and
# .max() on an all-NaN group correctly stays NaN rather than dropping the player.
furthest_milestone_per_player = milestones.groupby('user_id')['max_milestone_completed'].max().reset_index(name='furthest_milestone')

# dropna=False keeps the "0 milestones completed" players as their own bucket instead of
# pandas silently excluding the NaN group.
milestone_distribution = furthest_milestone_per_player.groupby('furthest_milestone', dropna=False).agg(
    n_players=('user_id', 'nunique'),
).reset_index().sort_values('furthest_milestone', na_position='first')

#milestone_distribution

In [47]:
# hide-output 
# NaN (0 milestones completed) needs an explicit label - astype(str) would render it as the
# literal string 'nan', which reads as a bug rather than Omni's blank bucket.
plot_df = milestone_distribution.copy()
plot_df['furthest_milestone_label'] = plot_df['furthest_milestone'].apply(
    lambda x: '(none)' if pd.isna(x) else str(int(x))
)

fig = px.bar(
    plot_df,
    x='furthest_milestone_label',
    y='n_players',
    title='Milestone Distribution (per Player, Iteration) - rp_LOpsMilestoneComplete reproduction',
    width=1200,
    height=500,
    labels={'furthest_milestone_label': 'Furthest Milestone', 'n_players': 'N Players'},
    color_discrete_sequence=[_FEW_QUALITATIVE[0]],
)
fig.update_xaxes(categoryorder='array', categoryarray=plot_df['furthest_milestone_label'].tolist())
few_style(fig)
fig.show()

# hide-output 
## Questions I would like answered
- How long does it take to complete 16 milestones?
- How much rewards do they get every milestone?
- What type of orders where they completing?
- Why are more than 16 milestones? why is that number still moving?
- What type of orders are these players completing?


## Some insights
- Engagement is working across the board
- Economy
    - Energy 
    - Gems is more towards neutral but also some impact from bonus rewards
- Both balances are not significantly affected
- Monetisation flips on Aug 25
- Ads gets boost on bubble generator


In [48]:
# hide-output 
export_notebook_html(
# Exports the notebook to HTML for sharing and archival
    notebook_path='./tempgen.ipynb',
    output_path='./tempgen.html',
)

Saved to tempgen.html


PosixPath('tempgen.html')